In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import matplotlib.pyplot as plt
import numpy as np
from src.semantic_taxonomy import assign_category, get_category_distribution, CATEGORIES

labels4  = torch.load('../checkpoints/labels_layer4.pt',  map_location='cpu', weights_only=False)
labels8  = torch.load('../checkpoints/labels_layer8.pt',  map_location='cpu', weights_only=False)
labels12 = torch.load('../checkpoints/labels_layer12.pt', map_location='cpu', weights_only=False)
print(f'Loaded: {len(labels4["labels"])} features per layer')

In [ ]:
cats4  = [assign_category(l) for l in labels4['labels']]
cats8  = [assign_category(l) for l in labels8['labels']]
cats12 = [assign_category(l) for l in labels12['labels']]

torch.save({'layer4': cats4, 'layer8': cats8, 'layer12': cats12},
           '../checkpoints/semantic_categories.pt')

print(f"{'Category':<14} {'Layer 4':>10} {'Layer 8':>10} {'Layer 12':>10}")
print('-' * 48)
d4, d8, d12 = [get_category_distribution(c) for c in [cats4, cats8, cats12]]
for cat in CATEGORIES:
    print(f'{cat:<14} {d4[cat]:>10} {d8[cat]:>10} {d12[cat]:>10}')

In [ ]:
dists = [d4, d8, d12]
layer_names = ['Layer 4', 'Layer 8', 'Layer 12']
cat_colors = {
    'background':  '#999999',
    'texture':     '#4e79a7',
    'color':       '#b07aa1',
    'object_part': '#f28e2b',
    'scene':       '#59a14f',
    'object':      '#e15759',
}

n_features = len(labels4['labels'])
fig, ax = plt.subplots(figsize=(9, 6))
bottoms = np.zeros(3)

for cat in CATEGORIES:
    counts = np.array([d[cat] / n_features * 100 for d in dists])
    ax.bar(layer_names, counts, bottom=bottoms,
           label=cat, color=cat_colors[cat], width=0.5)
    bottoms += counts

ax.set_ylabel('% of SAE features')
ax.set_title('Semantic Category Distribution Across Layers\n(Raghu 2021 hypothesis: Layer 4 should have more scene-level features than Layer 12)')
ax.legend(loc='upper right', bbox_to_anchor=(1.25, 1))
plt.tight_layout()
plt.savefig('../checkpoints/semantic_taxonomy_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def entropy(dist: dict, n_total: int) -> float:
    import math
    return -sum((v / n_total) * math.log2(v / n_total + 1e-9)
                for v in dist.values() if v > 0)

print(f"\n{'Layer':<12} {'Entropy (bits)':>16}")
print('-' * 30)
for name, dist in [('Layer 4', d4), ('Layer 8', d8), ('Layer 12', d12)]:
    h = entropy(dist, n_features)
    print(f'{name:<12} {h:>16.3f}')
print('(Higher entropy = more diverse feature types)')